In [15]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import math

<h2> Window-parting and Window-reversing </h2>

In [16]:
def part_window(x : torch.Tensor, window_size : int) -> torch.Tensor:
  '''
  Input:
  x:(torch.Tensor) input grid (B, H, W, C)
  window_size: (int) (w*w) w of the window size

  Return:
  x: (torch.Tensor) (B*h//window_size*w//window_size, window_size, window_size, C)

  '''
  #x -> (B, H, W, C)
  B, H, W, C = x.size()
  x = x.view(B, H // window_size, window_size, W // window_size, window_size, C)
  x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size, window_size, C)

  return x

In [17]:
def part_window_reverse(x : torch.Tensor, window_size : int, height : int, width : int) -> torch.Tensor:
  '''
  Input:
  x : (torch.tensor) (B*h//window_size*w//window_size, window_size, window_size, C)
  window_size: (int) size of window
  height: (int) height of the image
  width: (int) width of the image
  '''
  C = x.shape[3]
  x = x.view(-1, height // window_size, width // window_size, window_size, window_size, x.shape[3]).permute(0, 1, 3, 2, 4, 5).contiguous()
  B = x.shape[0]
  x = x.view(B, height, width, C) # (B, H, W, C)

  return x

In [18]:
x = torch.randn(8, 224, 224, 3)

window = part_window(x, 16)

print(window.shape)

torch.Size([1568, 16, 16, 3])


In [19]:
class MHSA(nn.Module):
  def __init__(self, n_heads, d_model):
    super().__init__()
    self.n_heads = n_heads
    self.d_model = d_model
    self.head = self.d_model // self.n_heads

    self.qkv = nn.Linear(d_model, d_model * 3, bias = False)
    self.proj = nn.Linear(d_model, d_model)


  def forward(self, x):
    B, N, D = x.shape
    qkv = self.qkv(x)
    qkv = qkv.view(B, N, 3, self.n_heads, self.head).permute(2, 0, 3, 1, 4).contiguous()
    q, k, v = qkv[0], qkv[1], qkv[2] #(B, H, N, D)

    wei = torch.matmul(q, k.transpose(-2, -1)) / self.head ** 0.5
    wei = torch.softmax(wei, dim = -1)

    att = torch.matmul(wei, v)

    att = att.permute(0, 2, 1, 3).contiguous().view(B, N, D)

    return self.proj(att)





<h2>Windowed Self-Attention </h2>

In [20]:
class WindowMSA(nn.Module):
  def __init__(self, n_heads : int, d_model : int, window_size : int, bias : bool = False, atn_dropout: float = 0.2, projection_dropout : float = 0.2):
    super().__init__()
    self.n_heads = n_heads
    self.d_model = d_model
    self.window_size = window_size
    self.head_dim = self.d_model // self.n_heads

    #relative bias (2*M-1, 2*M-1, n_heads)
    self.rel_bias_tab = nn.Parameter(torch.zeros((2 * window_size - 1) * (2 * window_size - 1), n_heads))

    #relative position index
    coords_h = torch.arange(window_size) #(window_size)
    coords_w = torch.arange(window_size) #(window_size)
    coords = torch.stack(torch.meshgrid([coords_h, coords_w], indexing = "ij")) # (2, window_size, window_size)
    coords_flat = coords.view(coords.size(0), -1) # (2, N) # (N = window_size * window_size)
    rel_coords = coords_flat[:, :, None] -  coords_flat[:, None, :] # (2, N, N) (2 co-ordinates and distances between all in the co-ordinate space)
    rel_coords = rel_coords.permute(1, 2, 0).contiguous() # (N,  N,  2) ( 2 are the indices of x diff and y diff)
    rel_coords[:, :, 0] += window_size - 1 # -1, 0, 1 wy convert to 0, 1, 2 (co-ordinates can't be negative)
    rel_coords[:, :, 1] += window_size - 1
    rel_coords[:, :, 0] *= 2 * window_size - 1 # FOR (2,2) window multipsy by 3 #to map  (0, 0) sum etc to unique 4 numbers

    #weights init
    nn.init.trunc_normal_(self.rel_bias_tab, std = 0.02)

    #N, N
    rel_position_index = rel_coords.sum(-1) # (0,1) => 1, (0,0) => 0 (3,0) (0,2) => 2 (3,0) => 3.(w.r.t each boxy we have 4 indices in this case to be compared)
    self.register_buffer('rel_position_index', rel_position_index)


    #projecgion and attention
    self.qkv = nn.Linear(d_model, d_model * 3, bias)
    self.proj = nn.Linear(d_model, d_model)
    self.atn_dropout = nn.Dropout(atn_dropout)
    self.projection_dropout = nn.Dropout(projection_dropout)


  def forward(self, x : torch.Tensor, mask : torch.Tensor = None) -> torch.Tensor:
      '''
      x: torch.Tensor(B_windows, N, C)
      N: window_size * window_size (all patches)
      B_windows: batch of all windows
      C: embedding dim of each patch
      mask: mask for shifted window self-attention (nW, N, N)
      '''
      B_dim, N, C = x.size()
      qkv = self.qkv(x) # (B*h//window_size*w//window_size, N * N, C*3)
      #(3, B_win,  n_heads, N, head_dim)
      qkv = qkv.view(B_dim, N, 3, self.n_heads, self.head_dim).permute(2, 0, 3, 1, 4).contiguous()
      q, k, v = qkv[0], qkv[1], qkv[2] # (B_win, n_heads, N, head_dim)

      #att weight
      wei = torch.matmul(q, k.transpose(-2, -1)) / (self.head_dim ** 0.5) # (B_windows, n_heads, N, N)
      #position bias
      #N, N, n_heads
      rel_pos_bias = self.rel_bias_tab[self.rel_position_index.view(-1)].view(N, N, -1)
      rel_pos_bias = rel_pos_bias.permute(2, 0, 1).contiguous() # n_heads, N, N
      wei = wei + rel_pos_bias.unsqueeze(0) # (B_win, n_heads, N, N)


      #shifted windows

      if mask is not None:
        #mask : (nW, N, N)
        nw = mask.size(0)
        #(B, nw, n_heads, N,N)
        wei = wei.view(B_dim // nw, nw, self.n_heads, N, N) + mask.unsqueeze(1).unsqueeze(0)
        wei = wei.view(-1, self.n_heads, N, N)
        wei = F.softmax(wei, dim = -1) #(B_win, n_heads, N, N)

      else:
        wei = F.softmax(wei, dim = -1) #(B_win, n_heads, N, N)

      wei = self.atn_dropout(wei)
      #(Bwin, n_heads, N, head_dim) => (B_windows, N, C)
      atn = (wei @ v).transpose(1, 2).contiguous().view(B_dim, N , C)
      atn = self.proj(atn)
      atn = self.projection_dropout(atn)

      return atn










In [21]:
wsa = WindowMSA(8, 512, 8)

print(wsa)

WindowMSA(
  (qkv): Linear(in_features=512, out_features=1536, bias=False)
  (proj): Linear(in_features=512, out_features=512, bias=True)
  (atn_dropout): Dropout(p=0.2, inplace=False)
  (projection_dropout): Dropout(p=0.2, inplace=False)
)


In [22]:
x = torch.randn(16, 8 * 8, 512)
out = wsa(x)

print(x.shape)


torch.Size([16, 64, 512])


<h2>MLP</h2>

In [23]:
class MLP(nn.Module):
  def __init__(self, in_dim : int, hidden_dim : int = None, out_dim : int = None, dropout : float = 0.3):
    super().__init__()
    out_dim  = out_dim if out_dim is not None else in_dim
    self.fc1 = nn.Linear(in_dim, hidden_dim, bias = True)
    self.act = nn.GELU()
    self.fc2 = nn.Linear(hidden_dim, out_dim, bias = True)
    self.dropout = nn.Dropout(dropout)


  def forward(self, x : torch.Tensor) -> torch.Tensor:
    x = self.act(self.fc1(x))
    out = self.dropout(self.fc2(x))
    return out

<h2>Swin-Transformer Block </h2>

In [27]:
class SwinTransformerBlock(nn.Module):
  def __init__(self, n_heads : int, d_model : int, inp_resolution : tuple[int, int], window_size : int = 7, shift_size : int = 0,
               mlp_ratio : float = 4, atn_dropout: float = 0.2, projection_dropout : float = 0.2, mlp_dropout : float = 0.2):

    super().__init__()

    self.d_model = d_model
    self.inp_resolution = inp_resolution # (H, W)
    self.n_heads = n_heads
    self.window_size = window_size
    self.shift_size = shift_size
    self.mlp_ratio = mlp_ratio

    #no shift if image is smaller than window size
    #also preprogram to just make window size equal to the shortest dimension if it exceeds any dimensions.
    if min(inp_resolution) <= window_size:
      self.shift_size = 0
      self.window_size = min(self.inp_resolution)

    #shift size is to be in [0, window_size)
    assert 0 <= self.shift_size < self.window_size, "shift size has to be in [0, window_size)"


    #layer architecture
    self.ln1 = nn.LayerNorm(d_model)
    self.atn = WindowMSA(n_heads, d_model, self.window_size, False, atn_dropout, projection_dropout)

    self.ln2 = nn.LayerNorm(d_model)
    hid_dim = int(d_model * mlp_ratio)
    self.mlp = MLP(d_model, hidden_dim = hid_dim, out_dim = d_model, dropout = mlp_dropout)

    #IF SHIFT
    if self.shift_size > 0:
      atn_mask = self.generate_mask(self.inp_resolution)

    else:
      atn_mask = None

    self.register_buffer('atn_mask', atn_mask)

   #function to calculate mask
  def generate_mask(self, res : tuple[int, int]) -> torch.Tensor:
        H, W = res
        mask = torch.zeros(1, H, W, 1)

        #slices
        h_slice = (
            slice(0, -self.window_size),
            slice(-self.window_size, -self.shift_size),
            slice(-self.shift_size, None)
        )

        w_slice = (
            slice(0, -self.window_size),
            slice(-self.window_size, -self.shift_size),
            slice(-self.shift_size, None)
        )

        cnt = 0

        #give slices regions (1, H, W, 1)
        for h in h_slice:
          for w in w_slice:
              mask[:, h, w, :] = cnt
              cnt += 1

        #partition the mask into windows
        win_mask = part_window(mask, self.window_size) #(h*w/window**2, window_sz, window_sz, 1)
        win_mask = win_mask.view(-1, self.window_size * self.window_size) #(n_windows, N)

        #attention mask
        atn_mask = win_mask.unsqueeze(1) - win_mask.unsqueeze(2) # (n_windows, N, N) each patches indiv difference with the other patches

        #plaicing large negative (-100) where mask is non-zero, that means different group and rest 0 (same)
        atn_mask = atn_mask.masked_fill(atn_mask != 0, float(-100)).masked_fill(atn_mask == 0, float(0)) #(n_windows, N, N)

        return atn_mask


  def forward(self, x : torch.Tensor) -> torch.Tensor:
        H, W = self.inp_resolution
        B, T, C = x.shape
        assert T == H * W, "Input resolution and model resolution must match"

        dup = x
        x = self.ln1(x)
        x = x.view(B, H, W, C) # (Batch, H, W, C)

        #if shifting
        if self.shift_size > 0:
            x_shift = torch.roll(x, shifts = (-self.shift_size, -self.shift_size), dims = (1, 2))

        else:
            x_shift = x

        #part window
        x_window = part_window(x_shift, self.window_size) # (B * h*w// window_size**2, window_size, window_size, C)
        B_win = x_window.size(0)
        x_window = x_window.view(-1, self.window_size * self.window_size, C)

        #masked attention
        atn_x = self.atn(x_window, mask = self.atn_mask) # (B_windows, N, C)
        atn_x = atn_x.view(B_win, self.window_size, self.window_size, C) # (B_windows, window_sz, window_sz, C)

        #reverse window partition
        x_shift = part_window_reverse(atn_x, self.window_size, H, W) # (B, H, W, C)

        #reversing cyclic shifting
        if self.shift_size > 0:
          x = torch.roll(x_shift, shifts = (self.shift_size, self.shift_size), dims = (1, 2))

        else:
          x = x_shift


        x = x.view(B, H * W, C)

        #residual
        x =  dup + x

        #(B, T, C)
        x = x + self.mlp(self.ln2(x))


        return x











In [25]:
x = torch.randn(2, 1)
y = torch.randn(1, 2)

print(x)
print(y)

y - x

tensor([[-0.1198],
        [ 0.7800]])
tensor([[1.8867, 1.8371]])


tensor([[2.0064, 1.9569],
        [1.1066, 1.0571]])

<h2>Patch-Merging </h2>

Reduces resolution by 2. H/2 and W/2 and doubles the channels.

Technique.
Sample H/2 and W/2 from top-left, bottom-right, top-right and bottom-left. tAKE ALTERNATE PIXELS.

Then concat to H/2, W/2 AND 4C. After that authors apply layer norm and project to 2C

In [31]:
class PatchMerging(nn.Module):
  def __init__(self, inp_resolution : tuple[int, int], d_embed : int):
    super().__init__()
    self.inp_resolution = inp_resolution
    self.d_embed = d_embed
    self.ln1 = nn.LayerNorm(d_embed * 4)
    self.downsample = nn.Linear(in_features = 4 * d_embed, out_features = 2 * d_embed, bias = False)


  def forward(self, x : torch.Tensor) -> torch.Tensor:
    B, T, C = x.shape
    H, W = self.inp_resolution

    assert H * W == T, "input dimensions must be equal to models resolution parameter."
    assert H % 2 == 0 and W % 2 ==0, "Spatial dimensions must be even"

    x = x.view(B, H, W, C)

    tl = x[:, :: 2, :: 2, :]
    tr = x[:, ::2, 1::2, :]
    bl = x[:, 1::2, ::2, :]
    br = x[:, 1::2, 1::2, :]

    #(B, H/2, W/2, 4)
    x_comb = torch.cat((tl, bl, tr, br), dim = -1)
    x_comb = x_comb.view(B, -1, 4 * C)

    x_comb = self.ln1(x_comb)
    #(B, H/W, W/2, 2)
    x_comb = self.downsample(x_comb)

    return x_comb






In [33]:
pm = PatchMerging((224, 224), 512)

x = torch.randn(4, 224 *224, 512)

x_out = pm(x)

print(x_out.shape)

torch.Size([4, 12544, 1024])
